# Quick GRPO Training (30 Episodes, ~15 min)
**Meta PyTorch OpenEnv Hackathon 2026**

Lightweight 30-episode run. 1 rollout, 1 pass, 6 gradient steps.
Forced curriculum: 6 eps per tier x 5 tiers.

Runtime -> Change runtime type -> **T4 GPU** first.

In [ ]:
# Cell 1 — Check GPU
import subprocess, torch
r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip() or '❌ No GPU — go to Runtime → Change runtime type → T4')
print('CUDA available:', torch.cuda.is_available())

In [ ]:
# Cell 2 — Install packages
%%capture
!pip install -q openenv-core "transformers>=4.45.0" "peft>=0.13.0" "bitsandbytes>=0.43.0" "accelerate>=0.34.0" wandb pydantic numpy python-dotenv
print('done')

In [ ]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
import os
SAVE_DIR = '/content/drive/MyDrive/the_pivot_model'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'✅ Saving to: {SAVE_DIR}')

In [ ]:
# Cell 4 — Clone project
import os, sys
!rm -rf /content/meta_scaler
!git clone https://github.com/Harshit-Makraria/meta_scaler /content/meta_scaler
sys.path.insert(0, '/content/meta_scaler')
os.chdir('/content/meta_scaler')

try:
    from models import CoFounderAction, ActionType, CoFounderObservation
    from server.cofounder_environment import CoFounderEnvironment
    from server.prompt_encoder import encode_to_messages
    from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
    print('✅ All imports OK!')
    print('Actions available:', len(list(ActionType)))
    print('Difficulty ladder:', DIFFICULTY_LADDER)
except ImportError as e:
    print(f'❌ Import error: {e}')

In [ ]:
# Cell 5 — W&B login
import wandb
wandb.login()   # paste your key from wandb.ai/settings
WANDB_PROJECT = 'models-nexica-ai'
print('✅ W&B ready')

In [ ]:
# Cell 6 — Load model (Qwen2.5-1.5B + LoRA)
# Using 1.5B instead of 0.5B: 3x more parameters → better startup reasoning.
# 1.5B at 4-bit ≈ 2GB VRAM for weights + ref copy ≈ 5GB total → fits T4 (15GB).
import os, gc, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
DEVICE     = 'cuda'
MAX_SEQ_LEN = 512

# ── Token budget (IMPORTANT for speed) ───────────────────────────────────────
# TRAIN_MAX_TOKENS = 16  — used during training generation.
#   The model only needs to output "DECISION: execute" = ~4-6 tokens.
#   16 is the safe upper bound. Generating 64 tokens per step × 60 steps
#   = 3,840 wasted tokens per episode. At 4 min/ep that's ~10h total.
#   With 16 tokens: ~1 min/ep → 100 episodes ≈ 1.5-2h on T4. ✅
#
# DEMO_MAX_TOKENS = 200 — used in demo cells for full reasoning readout.
TRAIN_MAX_TOKENS = 16
DEMO_MAX_TOKENS  = 200

print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    ),
    device_map='auto',
    trust_remote_code=True,
)

model = get_peft_model(base_model, LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8, lora_alpha=16, lora_dropout=0.05,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
))

# Allows gradients to flow into LoRA adapters through frozen 4-bit base layers
model.enable_input_require_grads()

model.print_trainable_parameters()
gc.collect(); torch.cuda.empty_cache()

used  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n✅ Model ready!  GPU: {used:.2f} GB used / {total:.1f} GB total  ({total-used:.1f} GB free)')
print(f'   TRAIN_MAX_TOKENS={TRAIN_MAX_TOKENS}  DEMO_MAX_TOKENS={DEMO_MAX_TOKENS}')

In [ ]:
# Cell 7 — Helpers
import re, random, gc, numpy as np
from models import CoFounderAction, ActionType, CoFounderObservation
from server.cofounder_environment import CoFounderEnvironment
from server.prompt_encoder import encode_to_messages

# Full 12-action map — supports all CoFounder strategic actions
ACTION_MAP = {
    'execute':            ActionType.EXECUTE,
    'pivot':              ActionType.PIVOT,
    'research':           ActionType.RESEARCH,
    'fundraise':          ActionType.FUNDRAISE,
    'hire':               ActionType.HIRE,
    'cut_costs':          ActionType.CUT_COSTS,
    'cut':                ActionType.CUT_COSTS,
    'sell':               ActionType.SELL,
    'launch_feature':     ActionType.LAUNCH_FEATURE,
    'launch':             ActionType.LAUNCH_FEATURE,
    'marketing_campaign': ActionType.MARKETING_CAMPAIGN,
    'marketing':          ActionType.MARKETING_CAMPAIGN,
    'set_pricing':        ActionType.SET_PRICING,
    'pricing':            ActionType.SET_PRICING,
    'fire':               ActionType.FIRE,
    'partnership':        ActionType.PARTNERSHIP,
}

# Full list — all 12 actions
ACTION_LIST = list(ActionType)

# Weighted random pool for ε-greedy exploration.
# PIVOT is present but rare (~5%). LAUNCH_FEATURE and RESEARCH are common
# since the new env rewards product health heavily.
# At ε=0.6 with 60 steps: ~3-4 random PIVOTs per episode = healthy signal.
RANDOM_ACTIONS = (
    [ActionType.EXECUTE]            * 5 +   # 27.8% — most common safe action
    [ActionType.RESEARCH]           * 3 +   # 16.7%
    [ActionType.LAUNCH_FEATURE]     * 3 +   # 16.7% — important for PMF rewards
    [ActionType.FUNDRAISE]          * 2 +   # 11.1%
    [ActionType.HIRE]               * 1 +   #  5.6%
    [ActionType.CUT_COSTS]          * 1 +   #  5.6%
    [ActionType.MARKETING_CAMPAIGN] * 1 +   #  5.6%
    [ActionType.PARTNERSHIP]        * 1 +   #  5.6%
    [ActionType.PIVOT]              * 1     #  5.6% — rare but present for timing signal
)


def _parse(text: str) -> ActionType:
    """
    Parse action from model output.
    Handles two formats:
      1. Structured: looks for 'DECISION: ACTION_WORD' line (primary format)
      2. Simple: falls back to first word (old format / random actions)
    """
    for line in text.split('\n'):
        stripped = line.strip().lower()
        for prefix in ('decision:', 'recommendation:', 'action:'):
            if stripped.startswith(prefix):
                rest = stripped[len(prefix):].strip()
                word = re.sub(r'[^a-z_]', '', rest.split()[0]) if rest.split() else ''
                if word in ACTION_MAP:
                    return ACTION_MAP[word]
    # Fallback: first word of full output
    w = re.sub(r'[^a-z_]', '', text.lower().split()[0]) if text.strip() else 'execute'
    return ACTION_MAP.get(w, ActionType.EXECUTE)


@torch.no_grad()
def generate_action(obs, sector='b2b_enterprise', max_tokens=None):
    """
    Generate an action from the model.
    max_tokens defaults to TRAIN_MAX_TOKENS (16) during training.
    Pass DEMO_MAX_TOKENS for readable demo output.
    """
    if max_tokens is None:
        max_tokens = TRAIN_MAX_TOKENS   # fast during training — "DECISION: X" = ~5 tokens
    text = tokenizer.apply_chat_template(
        encode_to_messages(obs, sector=sector), tokenize=False, add_generation_prompt=True)
    inp  = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
    out  = model.generate(**inp, max_new_tokens=max_tokens,
                          do_sample=True, temperature=0.9, top_p=0.9,
                          pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.decode(out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    del inp, out
    return decoded, _parse(decoded)


def get_log_prob(prompt: str, completion: str) -> torch.Tensor:
    comp_ids = tokenizer(completion, return_tensors='pt',
                         add_special_tokens=False)['input_ids'][0]
    if comp_ids.shape[0] == 0:
        return torch.tensor(0.0, requires_grad=True, device=DEVICE)
    prompt_ids = tokenizer(prompt, return_tensors='pt',
                           truncation=True,
                           max_length=MAX_SEQ_LEN - comp_ids.shape[0])['input_ids'][0]
    full_ids = torch.cat([prompt_ids, comp_ids]).unsqueeze(0).to(DEVICE)
    p_len    = prompt_ids.shape[0]
    out         = model(input_ids=full_ids)
    comp_logits = out.logits[0, p_len - 1:-1, :].clone()
    target_ids  = full_ids[0, p_len:].clone()
    del out, full_ids
    torch.cuda.empty_cache()
    lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
    return lp[range(len(target_ids)), target_ids].mean()


def run_episode(scenario: dict, seed: int = 0, epsilon: float = 0.3) -> list[dict]:
    from training.market_data import infer_sector_from_scenario
    sector = infer_sector_from_scenario(scenario.get('name', 'b2b_enterprise'))
    env = CoFounderEnvironment(scenario=scenario, rng_seed=seed)
    obs = env.reset()
    steps = []
    for _ in range(60):
        prompt = tokenizer.apply_chat_template(
            encode_to_messages(obs, sector=sector), tokenize=False, add_generation_prompt=True)
        if random.random() < epsilon:
            action_type = random.choice(RANDOM_ACTIONS)   # instant — no model call
            completion  = action_type.value.upper()
        else:
            # TRAIN_MAX_TOKENS=16 keeps generation fast — "DECISION: X" = ~5 tokens
            completion, action_type = generate_action(obs, sector=sector)
        next_obs = env.step(CoFounderAction(action_type=action_type))
        steps.append({'prompt': prompt, 'completion': completion,
                      'action': action_type.value, 'reward': float(next_obs.reward or 0),
                      'step': next_obs.step, 'runway': next_obs.runway_remaining})
        obs = next_obs
        if obs.done: break
    return steps


# ── Sanity check ──────────────────────────────────────────────────────────────
model.train()
env  = CoFounderEnvironment()
obs  = env.reset()
_prompt = tokenizer.apply_chat_template(encode_to_messages(obs), tokenize=False, add_generation_prompt=True)
_lp = get_log_prob(_prompt, 'EXECUTE')
print(f'Gradient check:  log_prob={_lp.item():.4f}  requires_grad={_lp.requires_grad}')
assert _lp.requires_grad and _lp.grad_fn is not None, '❌ Gradients not flowing!'
model.eval()
print(f'✅ Helpers ready!  {len(ACTION_MAP)} action entries, TRAIN_MAX_TOKENS={TRAIN_MAX_TOKENS}')

In [ ]:
# Cell 8 — QUICK config: 30 episodes, minimal overhead, ~15 min on T4
from torch.optim import AdamW
from training.curriculum import AdaptiveCurriculum, DIFFICULTY_LADDER
import copy

CONFIG = {
    'n_episodes':        30,
    'eps_per_tier':      6,
    'lr':                1e-4,
    'grad_clip':         0.5,
    'grpo_steps_sample': 6,       # fewer steps per update = faster
    'save_every':        15,
    'log_every':         1,
    'kl_beta':           0.01,
    'adv_clip':          1.5,
    'reward_scale':      100.0,
}

optimizer  = AdamW(model.parameters(), lr=CONFIG['lr'])
curriculum = AdaptiveCurriculum(seed=42)

ref_model = copy.deepcopy(model)
ref_model.eval()
for p in ref_model.parameters():
    p.requires_grad_(False)

reward_baseline = 0.0
BASELINE_DECAY  = 0.85


def get_log_prob_nograd(prompt: str, completion: str) -> float:
    with torch.no_grad():
        comp_ids = tokenizer(completion, return_tensors='pt', add_special_tokens=False)['input_ids'][0]
        if comp_ids.shape[0] == 0:
            return 0.0
        prompt_ids = tokenizer(prompt, return_tensors='pt', truncation=True,
                               max_length=MAX_SEQ_LEN - comp_ids.shape[0])['input_ids'][0]
        full_ids = torch.cat([prompt_ids, comp_ids]).unsqueeze(0).to(DEVICE)
        p_len = prompt_ids.shape[0]
        out = ref_model(input_ids=full_ids)
        comp_logits = out.logits[0, p_len - 1:-1, :]
        target_ids = full_ids[0, p_len:]
        lp = torch.nn.functional.log_softmax(comp_logits, dim=-1)
        val = lp[range(len(target_ids)), target_ids].mean().item()
        del out, full_ids; torch.cuda.empty_cache()
        return val


def grpo_update(episode_steps: list, ep_reward: float) -> dict:
    global reward_baseline
    model.train()
    optimizer.zero_grad()

    scale = CONFIG['reward_scale']
    rewards = np.array([s['reward'] / scale for s in episode_steps], dtype=np.float32)
    baseline_norm = reward_baseline / scale
    adv = rewards - baseline_norm
    std = adv.std()
    if std > 1e-6:
        adv = adv / std
    adv = np.clip(adv, -CONFIG['adv_clip'], CONFIG['adv_clip'])
    reward_baseline = BASELINE_DECAY * reward_baseline + (1 - BASELINE_DECAY) * ep_reward

    sample_k = min(CONFIG['grpo_steps_sample'], len(episode_steps))
    abs_adv = np.abs(adv)
    probs = abs_adv / (abs_adv.sum() + 1e-8)
    if probs.sum() < 0.01:
        indices = random.sample(range(len(episode_steps)), sample_k)
    else:
        indices = np.random.choice(len(episode_steps), size=sample_k, replace=False, p=probs).tolist()

    total_loss = 0.0
    total_kl   = 0.0
    did_bwd    = False
    for idx in indices:
        sd = episode_steps[idx]
        lp  = get_log_prob(sd['prompt'], sd['completion'])
        ref = get_log_prob_nograd(sd['prompt'], sd['completion'])
        grpo_loss = (-lp * float(adv[idx])) / sample_k
        kl = (lp - ref) / sample_k
        loss = grpo_loss + CONFIG['kl_beta'] * kl
        if loss.requires_grad and loss.grad_fn is not None:
            loss.backward()
            did_bwd = True
        total_loss += grpo_loss.item()
        total_kl += kl.item()
        del lp, kl, loss
        gc.collect(); torch.cuda.empty_cache()

    if did_bwd:
        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
        optimizer.step()
    optimizer.zero_grad()
    model.eval()
    gc.collect(); torch.cuda.empty_cache()
    return {'loss': total_loss, 'kl': total_kl / sample_k}


print('QUICK config ready — ~15 min on T4')

In [ ]:
# Cell 9 — QUICK TRAIN: 30 episodes, 1 rollout, 1 pass, forced curriculum
import os
os.environ.pop('WANDB_MODE', None)

run = wandb.init(project=WANDB_PROJECT, name='grpo-qwen1.5b-quick-30ep',
                 config=CONFIG, tags=['grpo','qwen1.5b','cofounder','hackathon','quick'])
print(f'W&B: {run.url}')
print('-' * 80)

epsilon   = 0.50
EPS_DECAY = 0.96
EPS_MIN   = 0.10

episode_rewards, episode_lengths, all_losses = [], [], []
best_reward = float('-inf')
model.eval()
current_tier = 0
eps_per_tier = CONFIG['eps_per_tier']

for ep in range(1, CONFIG['n_episodes'] + 1):
    target_tier = min((ep - 1) // eps_per_tier, len(DIFFICULTY_LADDER) - 1)
    if target_tier != current_tier:
        current_tier = target_tier
        while curriculum.current_tier < current_tier:
            curriculum.advance_tier()
        print(f'\nTIER {current_tier+1}/5: {DIFFICULTY_LADDER[current_tier]}\n')

    scenario_name = DIFFICULTY_LADDER[current_tier]
    scenario = curriculum._all_scenarios.get(scenario_name)
    if not scenario:
        scenario = curriculum.sample_scenario()
        scenario_name = scenario.get('name', 'default')

    seed = ep * 17 + 3
    steps = run_episode(scenario, seed=seed, epsilon=epsilon)

    ep_reward   = sum(s['reward'] for s in steps)
    ep_length   = len(steps)
    survived    = ep_length >= 60
    pivot_count = sum(1 for s in steps if s['action'] == 'PIVOT')
    unique_acts = len(set(s['action'] for s in steps))

    episode_rewards.append(ep_reward)
    episode_lengths.append(ep_length)
    if ep_reward > best_reward:
        best_reward = ep_reward

    loss_stats = grpo_update(steps, ep_reward)
    all_losses.append(loss_stats['loss'])

    epsilon = max(EPS_MIN, epsilon * EPS_DECAY)

    gpu_gb = torch.cuda.memory_allocated() / 1e9
    mean6 = float(np.mean(episode_rewards[-6:])) if len(episode_rewards) >= 6 else float(np.mean(episode_rewards))
    status = 'SURVIVED' if survived else 'died'
    print(
        f'Ep {ep:2d}/30 | T{current_tier+1} {scenario_name:16s} | '
        f'r={ep_reward:+7.1f} | mean6={mean6:+7.1f} | '
        f'loss={loss_stats["loss"]:+7.4f} | eps={epsilon:.2f} | '
        f'piv={pivot_count} acts={unique_acts} | {status}'
    )

    wandb.log({
        'episode': ep, 'ep_reward': ep_reward, 'best_reward': best_reward,
        'mean_reward_6ep': mean6, 'ep_length': ep_length,
        'survived': int(survived), 'pivot_count': pivot_count,
        'unique_actions': unique_acts, 'loss': loss_stats['loss'],
        'kl': loss_stats.get('kl', 0), 'epsilon': epsilon,
        'curriculum_tier': current_tier + 1, 'scenario': scenario_name,
    })

    if ep % CONFIG['save_every'] == 0:
        ckpt = f'{SAVE_DIR}/checkpoint_ep{ep}'
        model.save_pretrained(ckpt); tokenizer.save_pretrained(ckpt)
        print(f'Checkpoint -> {ckpt}')

print('\n' + '='*72)
print('QUICK TRAINING COMPLETE!')
print(f'Mean reward last 6 ep  : {np.mean(episode_rewards[-6:]):.1f}')
print(f'Best reward            : {best_reward:.1f}')
print(f'Survival rate          : {sum(l>=60 for l in episode_lengths)/len(episode_lengths):.0%}')
print('='*72)

In [ ]:
# Cell 10 — Save model + close W&B
FINAL = f'{SAVE_DIR}/final_model'
model.save_pretrained(FINAL); tokenizer.save_pretrained(FINAL)
print(f'✅ Model saved to {FINAL}')
wandb.finish()
print('✅ W&B closed')

In [ ]:
# Cell 10c — 🚀 Push trained LoRA to HF Hub (enables chat UI on HF Space)
# ─────────────────────────────────────────────────────────────────────────────
# After running this cell:
#   1. Your LoRA weights are at huggingface.co/Harshit-Makraria/the-pivot-lora
#   2. Go to your HF Space → Settings → Variables and secrets
#      Add secret:  MODEL_ID = Harshit-Makraria/the-pivot-lora
#   3. Upgrade Space hardware to T4 small (Settings → Hardware → T4 small)
#   4. Rebuild Space — the chat panel will go live automatically
# ─────────────────────────────────────────────────────────────────────────────
from huggingface_hub import HfApi
import os

HF_USERNAME  = "Harshit-Makraria"           # your HF username
HF_REPO_NAME = "the-pivot-lora"             # will be created automatically
HF_MODEL_ID  = f"{HF_USERNAME}/{HF_REPO_NAME}"

# Paste your HF token (same one used to push the Space)
HF_TOKEN = input("Paste your HF token (hf_...): ").strip()

print(f"Pushing to {HF_MODEL_ID} ...")

# Push the LoRA adapter + tokenizer
model.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)
tokenizer.push_to_hub(HF_MODEL_ID, token=HF_TOKEN, private=False)

# Also write a model card
card = f"""---
base_model: Qwen/Qwen2.5-1.5B-Instruct
tags:
  - peft
  - lora
  - rl
  - grpo
  - startup
  - hackathon
license: mit
---

# The Pivot — RL-trained Startup Advisor (LoRA)

Fine-tuned with **GRPO** on the [The Pivot](https://github.com/Harshit-Makraria/meta_scaler) OpenEnv environment.

Trained to navigate hidden market phase shifts across 5 startup scenarios over 150 episodes.

## Usage

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = PeftModel.from_pretrained(base, "{HF_MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained("{HF_MODEL_ID}")
```

Built for the **Meta PyTorch OpenEnv Hackathon 2026**.
"""

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=card.encode(),
    path_in_repo="README.md",
    repo_id=HF_MODEL_ID,
    repo_type="model",
)

print(f"\n✅ Model live at: https://huggingface.co/{HF_MODEL_ID}")
print(f"\nNext steps:")
print(f"  1. Go to https://huggingface.co/spaces/Harshit-Makraria/the-pivot/settings")
print(f"  2. Add secret: MODEL_ID = {HF_MODEL_ID}")
print(f"  3. Upgrade hardware to T4 small")
print(f"  4. Click 'Factory reboot' — chat panel goes live in ~2 min")

In [ ]:
# Cell 10b — 🧪 CHAT WITH THE TRAINED MODEL
# ─────────────────────────────────────────────────────────────────────────────
# This cell proves the model is REALLY working — not just outputting random words.
# It runs a live episode and at each step shows:
#   • The exact situation (month, runway, revenue, NPS, churn, PMF, morale)
#   • What the model says (raw text output — should contain DECISION: line)
#   • What action was chosen (one of 12 actions)
#   • The reward received
#
# Compare early episodes (ep 1) vs late episodes (ep 150) — the reasoning improves.
# ─────────────────────────────────────────────────────────────────────────────
import json

def chat_demo(scenario_name='b2c_saas', seed=999, max_steps=60, verbose=True):
    """
    Run one episode with the trained model (ε=0 — pure model, no randomness).
    Print every step so you can see the model reasoning.
    """
    from training.curriculum import AdaptiveCurriculum
    c = AdaptiveCurriculum()
    scenario = c._all_scenarios.get(scenario_name)
    if not scenario:
        print(f'❌ Scenario "{scenario_name}" not found')
        return

    env = CoFounderEnvironment(scenario=scenario, rng_seed=seed)
    obs = env.reset()
    model.eval()

    total_reward = 0.0
    pivot_count  = 0
    action_log   = []

    print(f'\n{"="*70}')
    print(f'🎮  TRAINED MODEL DEMO  |  Scenario: {scenario_name}  |  Seed: {seed}')
    print(f'{"="*70}')
    print(f'{"Mo":>3} | {"Action":>16} | {"Reward":>7} | {"Run":>4} | {"Revenue":>10} | Output')
    print(f'{"-"*70}')

    for step in range(max_steps):
        # Build the prompt the model sees
        prompt_text = tokenizer.apply_chat_template(
            encode_to_messages(obs), tokenize=False, add_generation_prompt=True)

        # Ask the model — NO randomness (ε=0), pure policy
        inp = tokenizer(prompt_text, return_tensors='pt',
                        truncation=True, max_length=MAX_SEQ_LEN).to(DEVICE)
        with torch.no_grad():
            out = model.generate(
                **inp,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,            # greedy — deterministic model decision
                pad_token_id=tokenizer.eos_token_id,
            )
        raw_output = tokenizer.decode(
            out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        del inp, out
        torch.cuda.empty_cache()

        action_type = _parse(raw_output)
        next_obs    = env.step(CoFounderAction(action_type=action_type))
        reward      = next_obs.reward or 0.0
        total_reward += reward

        if action_type == ActionType.PIVOT:
            pivot_count += 1

        action_log.append({
            'month':   step + 1,
            'action':  action_type.value,
            'reward':  round(reward, 1),
            'runway':  next_obs.runway_remaining,
            'revenue': round(next_obs.monthly_revenue),
            'output':  raw_output[:40],
        })

        marker = ' ◀ PIVOT!' if action_type == ActionType.PIVOT else ''
        print(
            f'{step+1:>3} | {action_type.value:>16} | {reward:>+7.1f} | '
            f'{next_obs.runway_remaining:>3}mo | ${next_obs.monthly_revenue:>9,.0f} | '
            f'{raw_output[:30]}{marker}'
        )

        obs = next_obs
        if obs.done:
            break

    print(f'{"-"*70}')
    outcome = 'SURVIVED ✅' if obs.runway_remaining > 0 else 'BANKRUPT ❌'
    print(f'  Outcome : {outcome}')
    print(f'  Reward  : {total_reward:+.1f}')
    print(f'  Pivots  : {pivot_count}')
    print(f'  Steps   : {len(action_log)}')
    print(f'{"="*70}\n')
    return action_log


# ── Run demo on easy + hard scenario ─────────────────────────────────────────
print('Running demo on EASY scenario (b2c_saas) ...')
log_easy = chat_demo('b2c_saas', seed=999)

print('\nRunning demo on HARD scenario (fintech) ...')
log_hard = chat_demo('fintech', seed=999)

# ── Quick comparison vs untrained baseline ────────────────────────────────────
print('\n📊 What to look for to confirm learning:')
print('  1. DECISION: line appears in model output — format is correct')
print('  2. Model uses LAUNCH_FEATURE when PMF is low')
print('  3. Model uses MARKETING_CAMPAIGN only when PMF > 0.60')
print('  4. Model says CUT_COSTS / FIRE when runway < 6mo')
print('  5. Model says RESEARCH when churn is rising but NPS looks ok')
print('  6. Hard scenario (fintech) shows lower reward than easy (b2c_saas)')

In [ ]:
# Cell 12 — Save training plots as PNG (judges need these committed to the repo)
import matplotlib.pyplot as plt
import numpy as np, os, json, pathlib

PLOTS_DIR = pathlib.Path('/content/meta_scaler/docs/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# Also save to Drive for safekeeping
DRIVE_PLOTS = pathlib.Path(f'{SAVE_DIR}/plots')
DRIVE_PLOTS.mkdir(parents=True, exist_ok=True)

def _smooth(xs, k=10):
    if len(xs) < k: return xs
    return np.convolve(xs, np.ones(k)/k, mode='valid')

# ── Reward curve ──────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episode_rewards, alpha=0.3, color='tab:blue', label='per-episode')
if len(episode_rewards) >= 10:
    ax.plot(range(9, len(episode_rewards)), _smooth(episode_rewards, 10),
            color='tab:blue', linewidth=2, label='10-ep moving avg')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.set_xlabel('Episode'); ax.set_ylabel('Total reward')
ax.set_title('The Pivot — GRPO training reward curve (Qwen2.5-0.5B)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
for p in [PLOTS_DIR / 'reward_curve.png', DRIVE_PLOTS / 'reward_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight')
    print(f'  saved → {p}')
plt.close(fig)

# ── Loss curve ────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(all_losses, alpha=0.3, color='tab:red', label='per-episode loss')
if len(all_losses) >= 10:
    ax.plot(range(9, len(all_losses)), _smooth(all_losses, 10),
            color='tab:red', linewidth=2, label='10-ep moving avg')
ax.set_xlabel('Episode'); ax.set_ylabel('GRPO loss')
ax.set_title('The Pivot — GRPO training loss')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
for p in [PLOTS_DIR / 'loss_curve.png', DRIVE_PLOTS / 'loss_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight')
    print(f'  saved → {p}')
plt.close(fig)

# ── Survival / episode length ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(episode_lengths, color='tab:green', alpha=0.6)
ax.axhline(60, color='gray', linestyle='--', label='max (60)')
ax.set_xlabel('Episode'); ax.set_ylabel('Steps survived')
ax.set_title('The Pivot — episode length over training')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
for p in [PLOTS_DIR / 'survival_curve.png', DRIVE_PLOTS / 'survival_curve.png']:
    fig.savefig(p, dpi=120, bbox_inches='tight')
    print(f'  saved → {p}')
plt.close(fig)

# ── Dump raw metrics as JSON for the README ───────────────────────────
metrics = {
    'n_episodes': len(episode_rewards),
    'mean_reward_last_20': float(np.mean(episode_rewards[-20:])),
    'best_reward':         float(max(episode_rewards)),
    'survival_rate':       float(sum(l>=60 for l in episode_lengths) / len(episode_lengths)),
    'final_tier':          curriculum.current_tier + 1,
}
for p in [PLOTS_DIR / 'metrics.json', DRIVE_PLOTS / 'metrics.json']:
    with open(p, 'w') as f: json.dump(metrics, f, indent=2)
print('\n✅ All plots + metrics saved. Metrics:', metrics)
print('Now git push these:')
print('  cd /content/meta_scaler && git add docs/plots && git commit -m "Add training plots" && git push')

In [ ]:
# Cell 11 — Evaluate trained vs baselines (including StrategistAgent)
# ─────────────────────────────────────────────────────────────────────────────
# The trained model MUST beat StrategistAgent to prove it learned multi-factor
# nuance beyond rule-based heuristics. Per plan.md Section 8:
#   "The RL agent's strategic advice MUST outlive and out-earn this simple
#    baseline on average to prove it is learning advanced, multi-factor nuance."
# ─────────────────────────────────────────────────────────────────────────────
from training.baseline_agent import (
    RandomAgent, StubbornAgent, PanicAgent, StrategistAgent, run_episodes
)
import json

class TrainedAgent:
    name = 'trained_llm'
    def act(self, obs):
        _, at = generate_action(obs)
        return CoFounderAction(action_type=at)

c       = AdaptiveCurriculum()
agents  = [RandomAgent(), StubbornAgent(), PanicAgent(), StrategistAgent(), TrainedAgent()]
N_EVAL  = 20   # episodes per agent per scenario

print(f'{"Scenario":22s} | {"Agent":14s} | {"Reward":>8} | {"Surv":>5} | {"PMF":>5} | {"Moral":>5} | {"Score":>6}')
print('-' * 80)

results = []
for name in DIFFICULTY_LADDER:
    sc = c._all_scenarios.get(name)
    if not sc:
        continue
    for agent in agents:
        r = run_episodes(agent, sc, N_EVAL)
        results.append(r)
        pmf     = r.get('mean_final_pmf', 0)
        morale  = r.get('mean_final_morale', 0)
        score   = r.get('mean_balanced_score', 0)
        marker  = ' ◀ BEST' if agent.name == 'trained_llm' else ''
        print(
            f'{name:22s} | {agent.name:14s} | '
            f'{r["mean_reward"]:>+8.1f} | {r["survival_rate"]:>4.0%} | '
            f'{pmf:>5.2f} | {morale:>5.2f} | {score:>6.1f}{marker}'
        )
    print()  # blank line between scenarios

# ── Aggregate summary — did trained beat strategist? ─────────────────────────
def _mean_field(results, agent_name, field):
    vals = [r[field] for r in results if r['agent'] == agent_name]
    return sum(vals) / len(vals) if vals else 0.0

print('='*80)
print('AGGREGATE (mean across all scenarios):')
print(f'{"Agent":14s} | {"Reward":>8} | {"Survival":>8} | {"PMF":>5} | {"Morale":>6} | {"Bal.Score":>9}')
print('-'*60)
for agent in agents:
    n  = agent.name
    r  = _mean_field(results, n, 'mean_reward')
    s  = _mean_field(results, n, 'survival_rate')
    p  = _mean_field(results, n, 'mean_final_pmf')
    m  = _mean_field(results, n, 'mean_final_morale')
    bs = _mean_field(results, n, 'mean_balanced_score')
    print(f'{n:14s} | {r:>+8.1f} | {s:>7.0%} | {p:>5.2f} | {m:>6.2f} | {bs:>9.1f}')

trained_score    = _mean_field(results, 'trained_llm',  'mean_balanced_score')
strategist_score = _mean_field(results, 'strategist',   'mean_balanced_score')
beat = '✅ YES — LLM beat StrategistAgent!' if trained_score > strategist_score else '❌ NO — needs more training'
print(f'\nDid trained_llm beat StrategistAgent?  {beat}')
print(f'  trained_llm balanced score : {trained_score:.1f}')
print(f'  strategist  balanced score : {strategist_score:.1f}')

with open(f'{SAVE_DIR}/eval_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'\n✅ Evaluation done — saved to {SAVE_DIR}/eval_results.json')